In [1]:
import numpy as np
import jax
import jax.numpy as jnp

import pickle
from pathlib import Path

from train import load_data
from test_model import load_config
from models import HybridODE


### First load all the data into a library

In [2]:
angle_lib = list(range(0, 21, 5))
data_lib = []
for angle in angle_lib:
    base_name = "data" + str(angle) + "deg"
    data = np.concat([data for data in load_data(processed_dir="processed_data", base_name=base_name)])
    data_lib.append(data)

### Load the trained basis models


In [3]:
all_models = []
for i in angle_lib:  # just use 0 and 20 deg models as basis
    config = load_config("config.yaml")
    base_name = "data" + str(i) + "deg"
    config["data"]["input_dir"] = "isaac_data/" + base_name
    params_path = Path("results") / base_name / "model_params.pkl"   
    with open(params_path, "rb") as fp:
        params = pickle.load(fp)    
    model = HybridODE(config)
    all_models.append([model, params])

Using basic MLP model
Using basic MLP model
Using basic MLP model
Using basic MLP model
Using basic MLP model


### Perform linear regression

In [44]:
# Load some necessary info from config
st_dim = 7
dt = config['data']['dt']
neural_states = config['model']['neural_states']
nz = len(neural_states)

data = data_lib[1]  # use 5 deg data as example
n_samples = data.shape[0]
# basis = [all_models[0]]
basis = [all_models[0], all_models[-1], all_models[1]]
n_basis = len(basis)

def compute_basis(initial_state, current_input, next_input):
    # loop in Python because basis is a Python list
    z_list = []
    for model, params in basis:
        next_state = model.rk4_step(
            initial_state, current_input, next_input, dt, params, training=False
        )
        z_list.append(next_state[jnp.array(neural_states)])
    return jnp.stack(z_list, axis=1)  # shape (nz, n_basis)

# --- process one sample ---
def predict_ensemble_one_sample(data_slice):
    initial_state = data_slice[:st_dim, 0]
    next_state = data_slice[:st_dim, 1]
    current_input = data_slice[st_dim:, 0]
    next_input = data_slice[st_dim:, 1]

    z_real = next_state[jnp.array(neural_states)]  # shape (nz,)
    z_basis = compute_basis(initial_state, current_input, next_input)  # (nz, n_basis)
    return z_basis, z_real

# --- vectorize over all samples ---
predict_all = jax.vmap(predict_ensemble_one_sample)
# --- jit compile ---
predict_all = jax.jit(predict_all)

Z_basis, Z_real = predict_all(data)  # or [:1000]

Theta = []
print(f"The basis have {n_basis} models.")
for n in range(nz):
    X = Z_basis[:, n, :]   # (N, 2)
    Y = Z_real[:, n]       # (N,)
    # closed-form least squares: theta = (X^T X)^(-1) X^T y
    theta = jnp.linalg.inv(X.T @ X) @ (X.T @ Y)
    Theta.append(theta)

    # prediction
    Y_pred = X @ theta  # shape (N,)

    # mean squared error
    mse = jnp.mean((Y - Y_pred) ** 2)
    # mean absolute error
    mae = jnp.mean(jnp.abs(Y - Y_pred))
    # Print error metrics
    print("Errors in fit for state", neural_states[n])
    print("MSE:", mse, "MAE:", mae)

Theta = jnp.stack(Theta, axis=0)  # (nz, n_basis)
print("Coefficients:", Theta)

The basis have 3 models.
Errors in fit for state 4
MSE: 0.0040455256 MAE: 0.03179253
Errors in fit for state 5
MSE: 0.0194217 MAE: 0.027104167
Errors in fit for state 6
MSE: 0.00928933 MAE: 0.0425666
Coefficients: [[ 0.7050781   0.40112305 -0.10351562]
 [ 0.37316895 -0.08168793  0.55163574]
 [ 0.5214844   0.26843262  0.21386719]]


In [ ]:
model, params = all_models[0]

# --- process one sample ---
def predict_ODE_one_sample(data_slice):
    initial_state = data_slice[:st_dim, 0]
    next_state = data_slice[:st_dim, 1]
    current_input = data_slice[st_dim:, 0]
    next_input = data_slice[st_dim:, 1]

    z_real = next_state[jnp.array(neural_states)]  # shape (nz,)
    next_state = model.rk4_step(initial_state, current_input, next_input, dt, params, training=False)
    z_pred = next_state[jnp.array(neural_states)]      
    return z_pred, z_real

# --- vectorize over all samples ---
predict_ODE_all = jax.vmap(predict_ODE_one_sample)
# --- jit compile ---
predict_ODE_all = jax.jit(predict_ODE_all)

Z_pred, Z_real = predict_ODE_all(data)  # or [:1000]

for n in range(nz):
    X = Z_pred[:, n]   # (N, 2)
    Y = Z_real[:, n]       # (N,)

    # mean squared error
    mse = jnp.mean((Y - X) ** 2)
    # mean absolute error
    mae = jnp.mean(jnp.abs(Y - X))
    # Print error metrics
    print("Errors in fit for state", neural_states[n])
    print("MSE:", mse, "MAE:", mae)

Errors in fit for state 4
MSE: 0.004455149 MAE: 0.029144268
Errors in fit for state 5
MSE: 0.021183196 MAE: 0.014743969
Errors in fit for state 6
MSE: 0.009936314 MAE: 0.042580824


In [14]:
Z_basis.shape

(617400, 3, 2)